# Student Mental Health Risk Prediction
## Step 2: Data Preprocessing & Pipeline Construction

### Objective
In this step, we transform raw student survey data into a clean, numerical format ready for Machine Learning algorithms.

Key Principles Followed:
1. **Prevent Data Leakage**: Preprocessing transformations (imputer statistics, scalers, encoders) must be **fit ONLY on the training set**, then used to **transform both train and test sets**.
2. **Modular Architecture**: We build Scikit-learn `Pipeline` and `ColumnTransformer` objects that can be saved and reused seamlessly in production.
3. **Structured Splitting**: 80% train / 20% test with target stratification to maintain class balance.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent

from src.data.load_data import load_raw_data, save_processed_data
from src.preprocessing.preprocess import (
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN,
    DROP_COLUMNS,
    get_preprocessor,
    split_data,
    get_feature_names
)

print("Modules imported successfully!")


Modules imported successfully!


### 1. Load Raw Dataset
We load the original dataset via `load_raw_data()`.

In [2]:
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "student_mental_health.csv"

df = load_raw_data(RAW_DATA_PATH)
print(f"Loaded raw dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head(3)

Loaded raw dataset with 2000 rows and 17 columns.


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium


### 2. Train-Test Split (80% / 20%)
> **Crucial Concept: Preventing Data Leakage**
> We split the data **before** calculating any imputation medians or standard scaling parameters. If we imputed or scaled on the entire dataset first, information from the test set would "leak" into the training set, giving falsely optimistic evaluation scores.

In [3]:
X_train, X_test, y_train, y_test = split_data(df, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

print("\nTarget distribution in Training Set (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTarget distribution in Test Set (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

X_train shape: (1600, 15) | y_train shape: (1600,)
X_test shape:  (400, 15)  | y_test shape:  (400,)

Target distribution in Training Set (%):
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64

Target distribution in Test Set (%):
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


### 3. Build ColumnTransformer Pipeline
We define our preprocessing pipeline:
- **Numerical Pipeline**:
  - `SimpleImputer(strategy='median')`: Fills missing values with the median of training data.
  - `StandardScaler()`: Standardizes features to mean=0 and variance=1.
- **Categorical Pipeline**:
  - `SimpleImputer(strategy='most_frequent')`: Fills missing categories with the mode.
  - `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`: Encodes categorical variables into binary columns.

In [4]:
preprocessor = get_preprocessor()
print("Preprocessor structure:")
preprocessor

Preprocessor structure:


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

### 4. Fit Preprocessor on Train Data & Transform
Notice we call `.fit_transform()` ONLY on `X_train`, and `.transform()` on `X_test`.

In [5]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

feature_names = get_feature_names(preprocessor)

print(f"Original feature count: {X_train.shape[1]}")
print(f"Preprocessed feature count: {X_train_preprocessed.shape[1]}")
print(f"Processed feature names ({len(feature_names)}):")
for idx, name in enumerate(feature_names, 1):
    print(f"  {idx:02d}. {name}")

Original feature count: 15
Preprocessed feature count: 23
Processed feature names (23):
  01. Age
  02. Year_of_Study
  03. Sleep_Hours
  04. Study_Hours_Per_Week
  05. GPA
  06. Social_Media_Hours
  07. Screen_Time_Hours
  08. Financial_Stress
  09. Family_Support
  10. Attendance_Rate
  11. Gender_Female
  12. Gender_Male
  13. Gender_Non-binary
  14. Physical_Activity_High
  15. Physical_Activity_Low
  16. Physical_Activity_Moderate
  17. Diet_Quality_Average
  18. Diet_Quality_Good
  19. Diet_Quality_Poor
  20. Part_Time_Job_No
  21. Part_Time_Job_Yes
  22. Previous_Mental_Health_History_No
  23. Previous_Mental_Health_History_Yes


### 5. Inspect the Preprocessed Output
Let's convert the resulting NumPy array into a readable pandas DataFrame to inspect the transformed values.

In [6]:
X_train_processed_df = pd.DataFrame(
    X_train_preprocessed,
    columns=feature_names,
    index=X_train.index
)

print("Transformed training data sample:")
display(X_train_processed_df.head(5))

print("\nVerify that there are zero missing values:")
print(f"Missing values remaining: {X_train_processed_df.isnull().sum().sum()}")

Transformed training data sample:


,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate,...,Physical_Activity_High,Physical_Activity_Low,Physical_Activity_Moderate,Diet_Quality_Average,Diet_Quality_Good,Diet_Quality_Poor,Part_Time_Job_No,Part_Time_Job_Yes,Previous_Mental_Health_History_No,Previous_Mental_Health_History_Yes
726,-0.063706,-0.471636,-1.404182,-0.238603,0.074214,0.148597,1.098992,0.713748,0.643008,-1.244652,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1384,0.875744,-1.232340,-1.676036,-1.330997,-0.976093,0.327697,0.194658,-0.842691,0.368687,-0.465696,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1579,-1.003157,-0.471636,0.408177,-0.516037,-1.034443,-0.299153,-0.830254,-0.305988,0.752736,0.772288,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1098,0.875744,-1.232340,1.404975,-0.082547,0.891120,0.775447,1.098992,0.821088,-0.948052,-0.215317,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
582,0.406019,0.289067,0.408177,0.663055,-0.275888,-0.388703,-0.347943,0.660077,0.697872,-1.453301,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0



Verify that there are zero missing values:
Missing values remaining: 0


### 6. Save Processed Train and Test Sets
For transparency and reproducibility, we save the processed training and test splits to `data/processed/`.

In [7]:
train_full = X_train_processed_df.copy()
train_full[TARGET_COLUMN] = y_train.values

X_test_processed_df = pd.DataFrame(
    X_test_preprocessed,
    columns=feature_names,
    index=X_test.index
)
test_full = X_test_processed_df.copy()
test_full[TARGET_COLUMN] = y_test.values

save_processed_data(train_full, "train_processed.csv")
save_processed_data(test_full, "test_processed.csv")

print("Saved 'train_processed.csv' and 'test_processed.csv' to data/processed/ successfully!")


Saved 'train_processed.csv' and 'test_processed.csv' to data/processed/ successfully!


### 7. Step 2 Summary & Transition to Step 3

**Key Achievements in Preprocessing:**
1.  **Zero Data Leakage**: Preprocessor parameters were learned strictly on `X_train`.
2.  **Handled Missingness**: Median imputation for numerical; mode imputation for categorical.
3.  **Encoded Categorical Variables**: Converted 5 categorical features into 13 one-hot columns.
4.  **Standardized Numerical Features**: Scaled 10 numerical features to zero mean and unit variance.
5.  **Saved Processed Data**: Clean CSV datasets saved in `data/processed/`.